# Homework Starter — Stage 05: Data Storage
Name: Yufei Qin

Date: 08/19/2026

Objectives:
- Env-driven paths to `data/raw/` and `data/processed/`
- Save CSV and Parquet; reload and validate
- Abstract IO with utility functions; document choices

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
!pip install pyarrow
# !pip install python-dotenv

   ---------------------------------------- 0.0/27.8 MB ? eta -:--:--
   ----- ---------------------------------- 3.7/27.8 MB 18.2 MB/s eta 0:00:02
   ------------- -------------------------- 9.7/27.8 MB 21.6 MB/s eta 0:00:01
   --------------------- ------------------ 14.9/27.8 MB 22.9 MB/s eta 0:00:01
   ----------------------------- ---------- 20.7/27.8 MB 23.8 MB/s eta 0:00:01
   ---------------------------------------  27.8/27.8 MB 26.3 MB/s eta 0:00:01
   ---------------------------------------- 27.8/27.8 MB 24.9 MB/s  0:00:01


In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: c:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework05\notebooks

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

RAW -> C:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework05\notebooks\data\raw
PROC -> C:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework05\notebooks\data\processed


## 1) Create or Load a Sample DataFrame
You may reuse data from prior stages or create a small synthetic dataset.

In [4]:
import numpy as np
dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({'date': dates, 'ticker': ['AAPL']*20, 'price': 150 + np.random.randn(20).cumsum()})
df.head()

,date,ticker,price
0,2024-01-01,AAPL,151.334512
1,2024-01-02,AAPL,151.032019
2,2024-01-03,AAPL,151.997194
3,2024-01-04,AAPL,151.168311
4,2024-01-05,AAPL,152.010262


## 2) Save CSV to data/raw/ and Parquet to data/processed/ (TODO)
- Use timestamped filenames.
- Handle missing Parquet engine gracefully.

In [6]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')


# Save CSV to data/raw/
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)

print("CSV saved to:")
print(csv_path.resolve())


# Save Parquet to data/processed/
pq_path = PROC / f"sample_{ts()}.parquet"

try:
    df.to_parquet(pq_path, index=False)
    print("\nParquet saved to:")
    print(pq_path.resolve())
except ImportError:
    print("\nParquet engine not available.")
    print("Please install pyarrow or fastparquet.")
    pq_path = None

CSV saved to:
C:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework05\notebooks\data\raw\sample_20260820-113536.csv

Parquet saved to:
C:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework05\notebooks\data\processed\sample_20260820-113536.parquet


## 3) Reload and Validate (TODO)
- Compare shapes and key dtypes.

In [7]:
def validate_loaded(original, reloaded):
    checks = {
        'shape_equal': original.shape == reloaded.shape,
        'columns_equal': original.columns.tolist() == reloaded.columns.tolist(),
        'date_is_datetime': (
            pd.api.types.is_datetime64_any_dtype(reloaded['date'])
            if 'date' in reloaded.columns else False
        ),
        'ticker_is_string': (
            pd.api.types.is_object_dtype(reloaded['ticker'])
            if 'ticker' in reloaded.columns else False
        ),
        'price_is_numeric': (
            pd.api.types.is_numeric_dtype(reloaded['price'])
            if 'price' in reloaded.columns else False
        ),
        'no_missing_values': reloaded.isna().sum().sum() == 0
    }
    
    return checks


df_csv = pd.read_csv(
    csv_path,
    parse_dates=['date']
)

csv_checks = validate_loaded(df, df_csv)

print("CSV validation:")
for check, result in csv_checks.items():
    print(f"{check}: {result}")

CSV validation:
shape_equal: True
columns_equal: True
date_is_datetime: True
ticker_is_string: True
price_is_numeric: True
no_missing_values: True


In [8]:
if pq_path is not None:
    try:
        df_pq = pd.read_parquet(pq_path)

        pq_checks = validate_loaded(df, df_pq)

        print("Parquet validation:")
        for check, result in pq_checks.items():
            print(f"{check}: {result}")

    except ImportError:
        print("Parquet engine not available. Please install pyarrow.")
else:
    print("Parquet file was not created.")

Parquet validation:
shape_equal: True
columns_equal: True
date_is_datetime: True
ticker_is_string: True
price_is_numeric: True
no_missing_values: True


## 4) Utilities (TODO)
- Implement `detect_format`, `write_df`, `read_df`.
- Use suffix to route; create parent dirs if needed; friendly errors for Parquet.

In [9]:
import typing as t
import pathlib


def detect_format(path: t.Union[str, pathlib.Path]):
    """
    Detect file format from file suffix.
    """
    s = str(path).lower()

    if s.endswith('.csv'):
        return 'csv'

    if s.endswith('.parquet') or s.endswith('.pq') or s.endswith('.parq'):
        return 'parquet'

    raise ValueError(f'Unsupported format: {path}')


def write_df(
    df: pd.DataFrame,
    path: t.Union[str, pathlib.Path]
):
    """
    Save DataFrame according to file suffix.
    """
    p = pathlib.Path(path)

    # Create parent directory automatically
    p.parent.mkdir(parents=True, exist_ok=True)

    fmt = detect_format(p)

    if fmt == 'csv':
        df.to_csv(p, index=False)

    elif fmt == 'parquet':
        try:
            df.to_parquet(p, index=False)

        except ImportError as e:
            raise RuntimeError(
                'Parquet engine not available. '
                'Install pyarrow or fastparquet.'
            ) from e

    return p


def read_df(
    path: t.Union[str, pathlib.Path]
):
    """
    Load DataFrame according to file suffix.
    """
    p = pathlib.Path(path)

    if not p.exists():
        raise FileNotFoundError(
            f'File not found: {p}'
        )

    fmt = detect_format(p)

    if fmt == 'csv':
        if 'date' in pd.read_csv(p, nrows=0).columns:
            return pd.read_csv(p, parse_dates=['date'])
        else:
            return pd.read_csv(p)

    elif fmt == 'parquet':
        try:
            return pd.read_parquet(p)

        except ImportError as e:
            raise RuntimeError(
                'Parquet engine not available. '
                'Install pyarrow or fastparquet.'
            ) from e


# -----------------------------
# Test utility functions
# -----------------------------

p_csv = RAW / f"util_{ts()}.csv"
p_pq = PROC / f"util_{ts()}.parquet"

write_df(df, p_csv)

print("Utility CSV saved:")
print(p_csv.resolve())

try:
    write_df(df, p_pq)

    print("\nUtility Parquet saved:")
    print(p_pq.resolve())

except RuntimeError as e:
    print(e)
    p_pq = None


# Read the files back
df_csv_util = read_df(p_csv)

print("\nCSV utility validation:")
print(validate_loaded(df, df_csv_util))


if p_pq is not None:
    df_pq_util = read_df(p_pq)

    print("\nParquet utility validation:")
    print(validate_loaded(df, df_pq_util))

Utility CSV saved:
C:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework05\notebooks\data\raw\util_20260820-113653.csv

Utility Parquet saved:
C:\Users\qwqqqyf\bootcamp_yufei_qin\homework\homework05\notebooks\data\processed\util_20260820-113653.parquet

CSV utility validation:
{'shape_equal': True, 'columns_equal': True, 'date_is_datetime': True, 'ticker_is_string': True, 'price_is_numeric': True, 'no_missing_values': np.True_}

Parquet utility validation:
{'shape_equal': True, 'columns_equal': True, 'date_is_datetime': True, 'ticker_is_string': True, 'price_is_numeric': True, 'no_missing_values': np.True_}


## 5) Documentation (TODO)
- Update README with a **Data Storage** section (folders, formats, env usage).
- Summarize validation checks and any assumptions.